# 🍅 Tomato Pipeline Demo

**YOLO Detection → Qwen3.5 Ripeness → Qwen3.5 Harvest Score Harvest Selection**

| Step | Model | Task |
|---|---|---|
| 1 | YOLO v2.6s (2-class) | 이미지에서 토마토 bbox 탐지 → 단일 `tomato` 클래스로 통합 |
| 2 | Qwen3.5-0.8B SFT (Ripeness) | 각 bbox crop → ripe / unripe 분류 |
| 3 | Qwen3.5-0.8B SFT (Harvest) | ripe 토마토 전체 이미지 + bbox → 최적 수확 대상 선정 |

**시각화 색상**
- ⬜ 흰색 : unripe (YOLO 탐지됐으나 미성숙)
- 🟥 빨간색 : ripe + harvest score 표시
- 🟩 라임색 : 최종 선택 토마토 (SELECTED)


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "5, 6"

import json
import re
import random
import time
from dataclasses import dataclass, field
from pathlib import Path

import cv2
import gradio as gr
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO
from unsloth import FastVisionModel

print("[INFO] Imports complete")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[INFO] Imports complete


In [2]:
# ── 경로 설정 ──────────────────────────────────────────────────────────────────
_ROOT = Path("/home/seoooa/project/tomato-agentic")

YOLO_WEIGHT         = str(_ROOT / "weights" / "yolo_v26s_twoclass.pt")
RIPENESS_MODEL_PATH = str(_ROOT / "notebook" / "final" / "weights" / "Tomato_Ripeness" / "VLLM_float16" / "qwen3.5_0.8b_lora")
HARVEST_MODEL_PATH  = str(_ROOT / "notebook" / "final" / "weights" / "Tomato_Harvest" / "VLLM_float16" / "qwen3_5_0_8b_lora")

HARVEST_TEST_DIR = str(_ROOT / "data" / "Tomato-Harvest-2k-Vision" / "test")
METADATA_PATH    = str(Path(HARVEST_TEST_DIR) / "metadata.jsonl")

YOLO_CONF = 0.7
YOLO_IOU  = 0.45
YOLO_DEVICE = 0   # YOLO GPU 번호 (CUDA_VISIBLE_DEVICES 기준)

print(f"[INFO] YOLO      : {YOLO_WEIGHT}")
print(f"[INFO] Ripeness  : {RIPENESS_MODEL_PATH}")
print(f"[INFO] Harvest   : {HARVEST_MODEL_PATH}")


[INFO] YOLO      : /home/seoooa/project/tomato-agentic/weights/yolo_v26s_twoclass.pt
[INFO] Ripeness  : /home/seoooa/project/tomato-agentic/notebook/final/weights/Tomato_Ripeness/VLLM_float16/qwen3.5_0.8b_lora
[INFO] Harvest   : /home/seoooa/project/tomato-agentic/notebook/final/weights/Tomato_Harvest/VLLM_float16/qwen3_5_0_8b_lora


In [3]:
# ── Ripeness 프롬프트 ──────────────────────────────────────────────────────────
RIPENESS_SYSTEM = """\
You are an expert tomato ripeness classifier.

Definitions:
- ripe: The center tomato shows any visible sign of ripening —
  fully red, predominantly red, or beginning to turn red
  (early blush, orange tint, reddish tint, or partial red coloration).
  It does NOT need to be fully red.

- unripe: The center tomato shows no sign of redness —
  it is fully green, yellow, or clearly pre-ripening.

Rules:
- Focus only on the CENTER tomato. Ignore borders, leaves, stems, and background.
- Base the decision on overall visible color, not tiny local artifacts.
"""

RIPENESS_USER = """\
Classify the CENTER tomato as ripe or unripe.

Return exactly one valid JSON object:
{
  "is_tomato": 1,
  "is_ripe": 0,
  "reasoning": ""
}

Rules:
- "is_tomato": 1 if a tomato is visible, else 0.
- "is_ripe": 1 if ripe, else 0.
- "reasoning": one sentence describing the visual evidence.
- Output only the JSON object, nothing else.
"""

RIPENESS_MAX_NEW_TOKENS = 96
RIPENESS_CROP_SIZE      = 512

# ── Harvest harvest score 프롬프트 ─────────────────────────────────────────────
HARVEST_SYSTEM = """\
You are a tomato harvest score prediction assistant.

Your task is to score ONE target tomato using two images:
1. The first image is the full scene image.
2. The second image is a crop image of the target tomato.

Input metadata gives the full image size, target tomato id, and target_bbox in full-image coordinates.

Score definitions:
- ripeness_score (0-10): judge only color maturity and redness of the target tomato.
- visibility_score (0-10): judge only how clearly the target tomato itself can be seen. Penalize leaves, stems, blur, darkness, glare, and border truncation. Do not penalize nearby tomatoes unless they occlude the target tomato.
- isolation_score (0-10): judge only tomato-to-tomato separation. Penalize touching, overlap, clustering, and crowding by other tomatoes. Do not penalize leaves, stems, blur, or color here.

Evidence usage:
- Use the crop image mainly for ripeness and visibility.
- Use the full image mainly for isolation and scene context.
- Score only the target tomato, not the whole image.
- Assign each criterion independently. Do not let a high score in one criterion compensate for another criterion.

Score anchors:
- 0-2: poor
- 3-4: low
- 5-6: moderate
- 7-8: good
- 9-10: excellent

Return only valid JSON. No markdown, no extra text.
"""

HARVEST_USER = """\
Score the target tomato.

Input:
{{
  "image_size": {image_size},
  "target_tomato_id": {target_tomato_id},
  "target_bbox": {target_bbox}
}}

Return exactly one valid JSON object:
{{
  "id": {target_tomato_id},
  "ripeness_score": 0,
  "visibility_score": 0,
  "isolation_score": 0
}}

Requirements:
- The output id must equal target_tomato_id.
- Scores must be integers from 0 to 10.
- Do not output total_score, selected_tomato_id, or reasoning.
- Output only the JSON object and nothing else.
"""

HARVEST_MAX_NEW_TOKENS = 80
HARVEST_CROP_SIZE      = 512
HARVEST_CROP_PADDING   = 20

print("[INFO] Prompts defined")


[INFO] Prompts defined


In [4]:
# ── YOLO ──────────────────────────────────────────────────────────────────────
if "yolo_model" not in dir() or yolo_model is None:
    yolo_model = YOLO(YOLO_WEIGHT)
    print(f"[INFO] YOLO loaded: {YOLO_WEIGHT}")

# ── Ripeness ──────────────────────────────────────────────────────────────────
if "ripeness_model" not in dir() or ripeness_model is None:
    ripeness_model, ripeness_tokenizer = FastVisionModel.from_pretrained(
        RIPENESS_MODEL_PATH, load_in_4bit=False
    )
    FastVisionModel.for_inference(ripeness_model)
    ripeness_tokenizer.padding_side = "left"
    print(f"[INFO] Ripeness model loaded: {RIPENESS_MODEL_PATH}")

# ── Harvest ───────────────────────────────────────────────────────────────────
if "harvest_model" not in dir() or harvest_model is None:
    harvest_model, harvest_tokenizer = FastVisionModel.from_pretrained(
        HARVEST_MODEL_PATH, load_in_4bit=False
    )
    FastVisionModel.for_inference(harvest_model)
    harvest_tokenizer.padding_side = "left"
    print(f"[INFO] Harvest model loaded: {HARVEST_MODEL_PATH}")


[INFO] YOLO loaded: /home/seoooa/project/tomato-agentic/weights/yolo_v26s_twoclass.pt
==((====))==  Unsloth 2026.4.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 2. Max memory: 47.413 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tokenizer you are loading from '/home/seoooa/project/tomato-agentic/notebook/final/weights/Tomato_Ripeness/VLLM_float16/qwen3.5_0.8b_lora' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


[INFO] Ripeness model loaded: /home/seoooa/project/tomato-agentic/notebook/final/weights/Tomato_Ripeness/VLLM_float16/qwen3.5_0.8b_lora
==((====))==  Unsloth 2026.4.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 2. Max memory: 47.413 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tokenizer you are loading from '/home/seoooa/project/tomato-agentic/notebook/final/weights/Tomato_Harvest/VLLM_float16/qwen3_5_0_8b_lora' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


[INFO] Harvest model loaded: /home/seoooa/project/tomato-agentic/notebook/final/weights/Tomato_Harvest/VLLM_float16/qwen3_5_0_8b_lora


In [5]:
# ════════════════════════════════════════════════════════════════════
#  Step 1 — YOLO Detection
# ════════════════════════════════════════════════════════════════════

@dataclass
class Detection:
    confidence: float
    xyxy: list[float] = field(default_factory=list)  # [x1, y1, x2, y2]

def step1_detect(pil_image: Image.Image) -> list[Detection]:
    """PIL 이미지에서 토마토를 탐지하고 단일 클래스로 통합합니다."""
    results = yolo_model.predict(
        source=pil_image,
        conf=YOLO_CONF,
        iou=YOLO_IOU,
        device=YOLO_DEVICE,
        verbose=False,
        save=False,
    )
    detections = []
    r0 = results[0]
    if r0.boxes is None or len(r0.boxes) == 0:
        return detections
    for xyxy, conf in zip(r0.boxes.xyxy.cpu().numpy(), r0.boxes.conf.cpu().numpy()):
        detections.append(Detection(confidence=float(conf), xyxy=[float(v) for v in xyxy]))
    detections.sort(key=lambda d: d.xyxy[0])  # x1 오름차순 → 왼쪽부터 id=1
    return detections


def _crop_bbox(
    pil_image: Image.Image,
    xyxy: list[float],
    pad: int = 4,
) -> tuple[Image.Image, tuple[int, int, int, int]]:
    """PIL 이미지에서 bbox crop (패딩 포함)."""
    W, H = pil_image.size
    x1 = max(0, int(xyxy[0]) - pad)
    y1 = max(0, int(xyxy[1]) - pad)
    x2 = min(W, int(xyxy[2]) + pad)
    y2 = min(H, int(xyxy[3]) + pad)
    return pil_image.crop((x1, y1, x2, y2)), (x1, y1, x2, y2)


# ════════════════════════════════════════════════════════════════════
#  Step 2 — Ripeness Classification
# ════════════════════════════════════════════════════════════════════

@dataclass
class RipenessResult:
    is_tomato: int
    is_ripe:   int
    reasoning: str
    raw: str = ""

    @property
    def label(self) -> str:
        if not self.is_tomato:
            return "not_tomato"
        return "ripe" if self.is_ripe else "unripe"

def _parse_json(raw: str):
    try:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if m:
            return json.loads(m.group())
    except Exception:
        pass
    return None

def step2_ripeness(
    pil_image: Image.Image,
    detections: list[Detection],
    batch_size: int = 8,
) -> list[tuple[tuple[int,int,int,int], float, RipenessResult | None]]:
    """탐지된 모든 bbox를 crop한 뒤 배치로 익음도를 분류합니다.

    Returns: [(bbox, det_conf, RipenessResult | None), ...]
    """
    crops, bboxes, confs = [], [], []
    for det in detections:
        crop, bbox = _crop_bbox(pil_image, det.xyxy)
        crops.append(crop.resize((RIPENESS_CROP_SIZE, RIPENESS_CROP_SIZE)))
        bboxes.append(bbox)
        confs.append(det.confidence)

    ripeness_tokenizer.padding_side = "left"
    msgs_template = [
        {"role": "system", "content": [{"type": "text", "text": RIPENESS_SYSTEM}]},
        {"role": "user",   "content": [{"type": "image"}, {"type": "text", "text": RIPENESS_USER}]},
    ]

    all_results: list[RipenessResult | None] = []
    for start in range(0, len(crops), batch_size):
        batch_imgs = crops[start : start + batch_size]
        texts = [
            ripeness_tokenizer.apply_chat_template(
                msgs_template, add_generation_prompt=True, enable_thinking=False
            )
            for _ in batch_imgs
        ]
        inputs = ripeness_tokenizer(
            batch_imgs, texts,
            padding=True, add_special_tokens=False, return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            out_ids = ripeness_model.generate(
                **inputs, max_new_tokens=RIPENESS_MAX_NEW_TOKENS, do_sample=False, use_cache=True
            )

        input_len = inputs["input_ids"].shape[1]
        for out in out_ids:
            raw = ripeness_tokenizer.decode(out[input_len:], skip_special_tokens=True).strip()
            parsed = _parse_json(raw)
            all_results.append(
                RipenessResult(
                    is_tomato=int(parsed.get("is_tomato", 1)),
                    is_ripe=int(parsed.get("is_ripe", 0)),
                    reasoning=parsed.get("reasoning", ""),
                    raw=raw,
                ) if parsed else None
            )

    return list(zip(bboxes, confs, all_results))


# ════════════════════════════════════════════════════════════════════
#  Step 3 — Harvest Harvest Score + Python Argmax Selection
# ════════════════════════════════════════════════════════════════════

@dataclass
class TomatoScore:
    id: int
    ripeness_score: int
    visibility_score: int
    isolation_score: int
    total_score: int
    bbox_area: int = 0
    raw: str = ""

@dataclass
class HarvestResult:
    selected_tomato_id: int
    tomato_scores: list[TomatoScore] = field(default_factory=list)
    raw: str = ""


def _bbox_area(bbox: list[int | float]) -> int:
    try:
        x1, y1, x2, y2 = bbox
        return max(0, int(x2) - int(x1)) * max(0, int(y2) - int(y1))
    except Exception:
        return 0


def _safe_int_score(value, default: int = 0) -> int:
    try:
        return max(0, min(10, int(value)))
    except Exception:
        return default


def _crop_harvest_candidate(pil_image: Image.Image, bbox: list[int | float]) -> Image.Image:
    W, H = pil_image.size
    x1, y1, x2, y2 = bbox
    x1 = max(0, int(x1) - HARVEST_CROP_PADDING)
    y1 = max(0, int(y1) - HARVEST_CROP_PADDING)
    x2 = min(W, int(x2) + HARVEST_CROP_PADDING)
    y2 = min(H, int(y2) + HARVEST_CROP_PADDING)
    x1, x2 = min(x1, x2), max(x1, x2)
    y1, y2 = min(y1, y2), max(y1, y2)
    if x2 <= x1:
        x2 = x1 + 1
    if y2 <= y1:
        y2 = y1 + 1
    crop = pil_image.crop((x1, y1, x2, y2)).convert("RGB")
    if HARVEST_CROP_SIZE and crop.size != (HARVEST_CROP_SIZE, HARVEST_CROP_SIZE):
        crop = crop.resize((HARVEST_CROP_SIZE, HARVEST_CROP_SIZE), Image.BICUBIC)
    return crop


def _parse_harvest_score(raw: str, fallback_id: int, bbox: list[int | float]) -> TomatoScore:
    parsed = _parse_json(raw) or {}
    try:
        tid = int(parsed.get("id", fallback_id))
    except Exception:
        tid = fallback_id

    ripeness_score = _safe_int_score(parsed.get("ripeness_score"), 0)
    visibility_score = _safe_int_score(parsed.get("visibility_score"), 0)
    isolation_score = _safe_int_score(parsed.get("isolation_score"), 0)
    total_score = ripeness_score + visibility_score + isolation_score

    return TomatoScore(
        id=tid,
        ripeness_score=ripeness_score,
        visibility_score=visibility_score,
        isolation_score=isolation_score,
        total_score=total_score,
        bbox_area=_bbox_area(bbox),
        raw=raw,
    )


def step3_harvest(
    pil_image: Image.Image,
    ripeness_results: list[tuple],
) -> HarvestResult | None:
    """Ripe tomatoes are scored one candidate at a time, then selected by max total_score; ties use larger bbox area."""
    ripe_tomatoes = [
        {"id": i + 1, "bbox": list(bbox)}
        for i, (bbox, _conf, result) in enumerate(ripeness_results)
        if result is not None and result.label == "ripe"
    ]
    if not ripe_tomatoes:
        return None

    full_img = pil_image.convert("RGB")
    all_images = []
    input_texts = []

    for tomato in ripe_tomatoes:
        crop = _crop_harvest_candidate(full_img, tomato["bbox"])
        prompt_text = HARVEST_USER.format(
            image_size=json.dumps(list(full_img.size)),
            target_tomato_id=json.dumps(tomato["id"]),
            target_bbox=json.dumps(tomato["bbox"]),
        )
        msgs = [
            {"role": "system", "content": [{"type": "text", "text": HARVEST_SYSTEM}]},
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "image"},
                    {"type": "text", "text": prompt_text},
                ],
            },
        ]
        input_texts.append(
            harvest_tokenizer.apply_chat_template(
                msgs, add_generation_prompt=True, enable_thinking=False
            )
        )
        all_images.extend([full_img, crop])

    harvest_tokenizer.padding_side = "left"
    inputs = harvest_tokenizer(
        images=all_images,
        text=input_texts,
        padding=True,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        out_ids = harvest_model.generate(
            **inputs,
            max_new_tokens=HARVEST_MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=getattr(harvest_tokenizer, "pad_token_id", None) or getattr(harvest_tokenizer, "eos_token_id", None),
        )

    input_len = inputs["input_ids"].shape[1]
    raw_outputs = [
        harvest_tokenizer.decode(out[input_len:], skip_special_tokens=True).strip()
        for out in out_ids
    ]
    scores = [
        _parse_harvest_score(raw, fallback_id=tomato["id"], bbox=tomato["bbox"])
        for raw, tomato in zip(raw_outputs, ripe_tomatoes)
    ]

    selected = max(scores, key=lambda s: (s.total_score, s.bbox_area))
    raw_summary = json.dumps(
        {"per_candidate_raw_outputs": {str(t["id"]): raw for t, raw in zip(ripe_tomatoes, raw_outputs)}},
        ensure_ascii=False,
    )
    return HarvestResult(
        selected_tomato_id=selected.id,
        tomato_scores=scores,
        raw=raw_summary,
    )

print("[INFO] Pipeline step functions defined")


[INFO] Pipeline step functions defined


In [6]:
# 3색 bbox 시각화 (PIL 기반)
# ⬜ 흰색 : unripe   🟥 빨간색 : ripe + score   🟩 라임 : SELECTED

_COLOR_UNRIPE   = "white"
_COLOR_RIPE     = "red"
_COLOR_SELECTED = "lime"
_FONT_PATH      = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
_VIS_LONG_SIDE  = 1100
_FONT_SIZE      = 13
_BOX_WIDTH      = 3
_SELECTED_WIDTH = 4
_LABEL_PAD      = 4


def _get_font(size: int = _FONT_SIZE):
    try:
        return ImageFont.truetype(_FONT_PATH, size)
    except Exception:
        return ImageFont.load_default()


def _resize_for_visualization(pil_image: Image.Image, long_side: int = _VIS_LONG_SIDE) -> tuple[Image.Image, float]:
    """Resize every visualization to a stable display scale before drawing."""
    img = pil_image.convert("RGB")
    w, h = img.size
    if max(w, h) <= 0:
        return img, 1.0
    scale = long_side / max(w, h)
    new_size = (max(1, int(round(w * scale))), max(1, int(round(h * scale))))
    if new_size == img.size:
        return img.copy(), scale
    return img.resize(new_size, Image.Resampling.LANCZOS), scale


def _scale_bbox(bbox, scale: float) -> list[int]:
    x1, y1, x2, y2 = bbox
    return [int(round(float(v) * scale)) for v in (x1, y1, x2, y2)]


def _draw_label(draw: ImageDraw.ImageDraw, text: str, x: int, y: int, font, image_size: tuple[int, int]):
    """검정 배경 + 노란 텍스트 라벨을 그립니다."""
    bb   = draw.textbbox((0, 0), text, font=font)
    tw   = bb[2] - bb[0]
    th   = bb[3] - bb[1]
    pad  = _LABEL_PAD
    tx   = min(max(0, x + pad), max(0, image_size[0] - tw - 2 * pad))
    ty   = y - th - 2 * pad
    if ty < 0:
        ty = min(max(0, y + pad), max(0, image_size[1] - th - 2 * pad))
    draw.rectangle([tx - pad, ty - pad, tx + tw + pad, ty + th + pad], fill="black")
    draw.text((tx, ty), text, fill="yellow", font=font)


def draw_pipeline_image(
    pil_image: Image.Image,
    ripeness_results: list[tuple],
    harvest_result: HarvestResult | None,
) -> Image.Image:
    """파이프라인 결과를 3색 bbox로 시각화한 PIL 이미지를 반환합니다."""
    vis, scale = _resize_for_visualization(pil_image)
    draw = ImageDraw.Draw(vis)
    font = _get_font()

    selected_id = harvest_result.selected_tomato_id if harvest_result else -1
    score_map   = (
        {s.id: s.total_score for s in harvest_result.tomato_scores}
        if harvest_result else {}
    )

    for i, (bbox, _conf, result) in enumerate(ripeness_results):
        tid   = i + 1
        label = result.label if result else "unknown"
        x1, y1, x2, y2 = _scale_bbox(bbox, scale)

        if tid == selected_id:
            color = _COLOR_SELECTED
            width = _SELECTED_WIDTH
            score = score_map.get(tid)
            tag   = f"id:{tid} SELECTED" + (f" score:{score}" if score is not None else "")
        elif label == "ripe":
            color = _COLOR_RIPE
            width = _BOX_WIDTH
            score = score_map.get(tid)
            tag   = f"id:{tid} ripe" + (f" score:{score}" if score is not None else "")
        else:
            color = _COLOR_UNRIPE
            width = _BOX_WIDTH
            tag   = f"id:{tid} {label}"

        draw.rectangle([x1, y1, x2, y2], outline=color, width=width)
        _draw_label(draw, tag, x1, y1, font, vis.size)

    return vis

print("[INFO] Visualization function defined")

[INFO] Visualization function defined


In [7]:
def run_pipeline(
    pil_image: Image.Image,
) -> tuple[Image.Image, str, str, str]:
    """이미지 1장에 대해 전체 파이프라인을 실행합니다.

    Returns:
        (annotated_image, ripeness_json_str, harvest_json_str, timing_str)
    """
    timings: dict[str, float] = {}

    # Step 1: YOLO detection
    t0 = time.perf_counter()
    detections = step1_detect(pil_image)
    timings["1. YOLO detection"] = time.perf_counter() - t0

    if not detections:
        empty    = json.dumps({"message": "탐지된 토마토 없음"}, ensure_ascii=False, indent=2)
        timing_str = _format_timings(timings, n_detected=0, n_ripe=0)
        return pil_image, empty, empty, timing_str

    # Step 2: Ripeness classification
    t0 = time.perf_counter()
    ripeness_results = step2_ripeness(pil_image, detections)
    timings["2. Qwen Ripeness"] = time.perf_counter() - t0

    n_ripe = sum(1 for _, _, r in ripeness_results if r and r.label == "ripe")

    # Step 3: Harvest selection
    t0 = time.perf_counter()
    harvest_result = step3_harvest(pil_image, ripeness_results)
    timings["3. Qwen Harvest"] = time.perf_counter() - t0

    # Visualization
    annotated = draw_pipeline_image(pil_image, ripeness_results, harvest_result)

    # JSON 출력 포맷
    ripeness_out = [
        {
            "id":        i + 1,
            "bbox":      list(bbox),
            "det_conf":  round(conf, 4),
            "label":     result.label if result else "parse_error",
            "is_ripe":   result.is_ripe if result else None,
            "reasoning": result.reasoning if result else None,
        }
        for i, (bbox, conf, result) in enumerate(ripeness_results)
    ]

    id_to_bbox = {e["id"]: e["bbox"] for e in ripeness_out}
    harvest_out = None
    if harvest_result:
        harvest_out = {
            "selected_tomato_id": harvest_result.selected_tomato_id,
            "selected_bbox":      id_to_bbox.get(harvest_result.selected_tomato_id),
            "tomato_scores": [
                {
                    "id":               s.id,
                    "ripeness_score":   s.ripeness_score,
                    "visibility_score": s.visibility_score,
                    "isolation_score":  s.isolation_score,
                    "total_score":      s.total_score,
                    "bbox_area":        s.bbox_area,
                }
                for s in harvest_result.tomato_scores
            ],
        }

    timing_str = _format_timings(
        timings,
        n_detected=len(detections),
        n_ripe=n_ripe,
    )

    return (
        annotated,
        json.dumps(ripeness_out, ensure_ascii=False, indent=2),
        json.dumps(harvest_out,  ensure_ascii=False, indent=2),
        timing_str,
    )


def _format_timings(timings: dict[str, float], n_detected: int, n_ripe: int) -> str:
    total = sum(timings.values())
    lines = ["⏱  Step-by-Step Timing\n" + "─" * 36]
    for name, sec in timings.items():
        lines.append(f"  {name:<22}  {sec:6.2f} s")
    lines.append("─" * 36)
    lines.append(f"  {'Total':<22}  {total:6.2f} s")
    lines.append("")
    lines.append(f"  Detected : {n_detected} tomatoes")
    lines.append(f"  Ripe     : {n_ripe} tomatoes")
    return "\n".join(lines)


print("[INFO] run_pipeline defined")


[INFO] run_pipeline defined


In [8]:
# ── Test set에서 랜덤 10개 샘플 로드 ──────────────────────────────────────────
random.seed(42)
with open(METADATA_PATH) as f:
    _all_samples = [json.loads(line) for line in f if line.strip()]
_sampled = random.sample(_all_samples, min(20, len(_all_samples)))

_sample_names = [
    f"Sample {i+1}:  {s['file_name'][:55]}"
    for i, s in enumerate(_sampled)
]

def _load_sample(name: str | None):
    if name is None:
        return None
    idx = _sample_names.index(name)
    return Image.open(str(Path(HARVEST_TEST_DIR) / _sampled[idx]["file_name"]))

print(f"[INFO] {len(_sampled)} test examples loaded")


# ── Gradio UI ─────────────────────────────────────────────────────────────────
def predict(pil_image):
    if pil_image is None:
        empty = "이미지를 입력하세요."
        return None, empty, empty, ""
    annotated, rip_json, hvs_json, timing_str = run_pipeline(pil_image)
    return annotated, rip_json, hvs_json, timing_str


IMAGE_PREVIEW_HEIGHT = 520
DEMO_CSS = """
.image-preview { max-height: 560px; }
.image-preview .image-container,
.image-preview .image-frame,
.image-preview img,
.image-preview canvas {
    max-height: 520px !important;
    object-fit: contain !important;
}
"""


with gr.Blocks(title="🍅 Tomato Pipeline Demo", css=DEMO_CSS) as demo:
    gr.Markdown(
        "# 🍅 Tomato Pipeline Demo\n"
        "**YOLO Detection → Qwen3.5 Ripeness → Qwen3.5 Harvest Selection**\n\n"
        "Tie-break: if total_score is tied, the larger bbox area is selected.\n\n"
        "드롭다운에서 샘플을 선택하거나 직접 이미지를 업로드하세요."
    )

    # ── 입력 영역 ──────────────────────────────────────────────────────────────
    sample_dd = gr.Dropdown(
        choices=_sample_names,
        label="Test Set Examples (랜덤 10개) — 선택하면 자동 로드",
        value=None,
    )

    with gr.Row():
        img_input = gr.Image(
            type="pil",
            label="입력 이미지 (업로드 또는 샘플 선택)",
            height=IMAGE_PREVIEW_HEIGHT,
            elem_classes=["image-preview"],
        )

    sample_dd.change(
        fn=_load_sample,
        inputs=sample_dd,
        outputs=img_input,
    )

    run_btn = gr.Button("▶  Run Pipeline", variant="primary", size="lg")

    # ── 출력 영역 ──────────────────────────────────────────────────────────────
    with gr.Row():
        img_output  = gr.Image(
            type="pil",
            label="Pipeline 결과 시각화",
            height=IMAGE_PREVIEW_HEIGHT,
            elem_classes=["image-preview"],
        )
        out_timing  = gr.Textbox(lines=12, label="⏱  Timing", interactive=False)

    with gr.Row():
        out_ripeness = gr.Textbox(lines=20, label="Step 2 · Ripeness 결과 (JSON)")
        out_harvest  = gr.Textbox(lines=20, label="Step 3 · Harvest 결과 (JSON)")

    run_btn.click(
        fn=predict,
        inputs=img_input,
        outputs=[img_output, out_ripeness, out_harvest, out_timing],
    )

demo.launch(share=True)


[INFO] 20 test examples loaded


/tmp/ipykernel_3699711/627833558.py:43: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="🍅 Tomato Pipeline Demo", css=DEMO_CSS) as demo:


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://57e57a9fa993ac9b01.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
